# Quantum Layers in a CNN: Interactive Playground
*Companion to the poster **Integrating Quantum Layers to Classical Convolutional Neural Networks** (work in progress).*

Everything runs on simulated qubits and tiny models, so it is for **learning**, not for claiming a quantum advantage.
**How to use:** click *Runtime → Run all*, then scroll down and play with the sliders. Setup takes about a minute.

| Section | What you can do |
|---|---|
| 1. One patch | Move four pixel sliders and watch a 4-qubit circuit turn them into four numbers |
| 2. Quantum filter on an image | See the quantum feature maps from the poster, and change the circuit |
| 3. Model playground | Put a quantum layer at the **input, hidden or output**, and race it against a classical twin |
| 4. Look at predictions | Check what both models say about individual test digits |

In [ ]:
# Colab already has torch, scikit-learn, matplotlib and ipywidgets. Only PennyLane needs installing.
!pip -q install pennylane

In [ ]:
import numpy as np, matplotlib.pyplot as plt
import torch, torch.nn as nn
import pennylane as qml
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

DIGITS = load_digits()                 # 8x8 handwritten digits (small, fast, no download)
IMGS, LABELS = DIGITS.images / 16.0, DIGITS.target
print("Ready. PennyLane", qml.__version__, "| torch", torch.__version__)

In [ ]:
# ---------- A tiny 4-qubit simulator in plain NumPy (used for the quantum FILTER) ----------
# Same circuit as the poster figures: encode with RY, then layers of (RY, RZ) + a CNOT ring, then measure <Z>.
NQ = 4

def _ry(a):
    c, s = np.cos(a / 2), np.sin(a / 2)
    return np.stack([np.stack([c, -s], -1), np.stack([s, c], -1)], -2)

def _rz(a):
    return np.diag([np.exp(-1j * a / 2), np.exp(1j * a / 2)])

def _apply(s, U, q):                                   # apply a 1-qubit gate U to qubit q (batched)
    s = np.moveaxis(s, q + 1, -1)
    s = s @ U.T if U.ndim == 2 else np.einsum('b...j,bij->b...i', s, U)
    return np.moveaxis(s, -1, q + 1)

def _cnot(s, c, t):                                    # controlled-NOT (batched)
    s = s.copy(); idx = [slice(None)] * s.ndim; idx[c + 1] = 1
    sub = s[tuple(idx)]; ax = t + 1 if t < c else t
    s[tuple(idx)] = np.flip(sub, axis=ax); return s

def filter_weights(seed=7, depth=2):                   # a FIXED random circuit, as in the original quanvolution idea
    return np.random.default_rng(seed).uniform(0, 2 * np.pi, (depth, NQ, 2))

def quantum_filter(patches, W, entangle=True):         # patches: (B, 4) pixel values in [0, 1]
    B = patches.shape[0]
    s = np.zeros((B,) + (2,) * NQ, complex); s[(slice(None),) + (0,) * NQ] = 1
    for q in range(NQ): s = _apply(s, _ry(np.pi * patches[:, q]), q)          # ENCODE
    for l in range(W.shape[0]):                                                # PROCESS
        for q in range(NQ): s = _apply(s, _rz(W[l, q, 0]) @ _ry(W[l, q, 1]), q)
        if entangle:
            for q in range(NQ): s = _cnot(s, q, (q + 1) % NQ)
    p = np.abs(s) ** 2                                                         # MEASURE <Z> on each qubit
    rest = tuple(range(1, NQ))
    return np.stack([p.take(0, axis=q + 1).sum(rest) - p.take(1, axis=q + 1).sum(rest) for q in range(NQ)], 1)

def to_patches(imgs):                                  # (N, 8, 8) -> (N*16, 4): every 2x2 patch
    N = imgs.shape[0]
    return imgs.reshape(N, 4, 2, 4, 2).transpose(0, 1, 3, 2, 4).reshape(N * 16, 4)

def quanv_images(imgs, W, entangle=True):              # (N, 8, 8) -> (N, 4 channels, 4, 4)
    out = quantum_filter(to_patches(imgs), W, entangle)
    return out.reshape(len(imgs), 4, 4, 4).transpose(0, 3, 1, 2)

def classical_filter_images(imgs, seed=7):             # classical "twin": a fixed random linear filter + tanh
    R = np.random.default_rng(seed + 1).normal(size=(4, 4)) * 1.5
    out = np.tanh(to_patches(imgs) @ R)
    return out.reshape(len(imgs), 4, 4, 4).transpose(0, 3, 1, 2)

## 1. One patch through a 4-qubit circuit
Each 2x2 patch of pixels is **encoded** into 4 qubits, **processed** by gates (optionally entangled with CNOTs), and **measured** into 4 numbers.
Move the pixel sliders. Then switch entanglement off and see how the outputs change.

In [ ]:
px = [widgets.FloatSlider(value=v, min=0, max=1, step=0.05, description=d, continuous_update=False)
      for v, d in [(0.9, 'top-left'), (0.2, 'top-right'), (0.1, 'bottom-left'), (0.7, 'bottom-right')]]
w1_depth = widgets.IntSlider(value=2, min=1, max=4, description='layers', continuous_update=False)
w1_ent = widgets.Checkbox(value=True, description='entanglement (CNOT)')
w1_seed = widgets.IntSlider(value=7, min=0, max=50, description='circuit seed', continuous_update=False)

def show_patch(a, b, c, d, depth, ent, seed):
    vals = np.array([[a, b, c, d]])
    out = quantum_filter(vals, filter_weights(seed, depth), ent)[0]
    fig, ax = plt.subplots(1, 2, figsize=(7, 2.6), gridspec_kw={"width_ratios": [1, 2]})
    ax[0].imshow(vals.reshape(2, 2), cmap="gray", vmin=0, vmax=1); ax[0].set_title("2x2 patch in"); ax[0].axis("off")
    ax[1].bar(range(4), out, color="#c2185b"); ax[1].set_ylim(-1, 1); ax[1].axhline(0, color="k", lw=.5)
    ax[1].set_xticks(range(4)); ax[1].set_xlabel("qubit measured"); ax[1].set_title("4 numbers out (<Z>)")
    plt.tight_layout(); plt.show()

ui1 = widgets.VBox([widgets.HBox(px[:2] + px[2:3]), widgets.HBox(px[3:] + [w1_depth]), widgets.HBox([w1_ent, w1_seed])])
display(ui1, widgets.interactive_output(show_patch, dict(a=px[0], b=px[1], c=px[2], d=px[3], depth=w1_depth, ent=w1_ent, seed=w1_seed)))

## 2. The quantum filter on a whole image
Slide a 2x2 window across a digit; each window becomes 4 numbers, giving **4 quantum feature maps** (the figure on the poster uses the default settings below).
Some channels look flat. That is normal for a random, untrained circuit.

In [ ]:
w2_digit = widgets.Dropdown(options=list(range(10)), value=3, description='digit')
w2_idx = widgets.IntSlider(value=0, min=0, max=30, description='example #', continuous_update=False)
w2_depth = widgets.IntSlider(value=2, min=1, max=4, description='layers', continuous_update=False)
w2_ent = widgets.Checkbox(value=True, description='entanglement (CNOT)')
w2_seed = widgets.IntSlider(value=7, min=0, max=50, description='circuit seed', continuous_update=False)

def show_feature_maps(digit, idx, depth, ent, seed):
    pool = IMGS[LABELS == digit]; img = pool[idx % len(pool)]
    fm = quanv_images(img[None], filter_weights(seed, depth), ent)[0]
    fig, ax = plt.subplots(1, 5, figsize=(11, 2.4))
    ax[0].imshow(img, cmap="gray"); ax[0].set_title("input 8x8")
    for k in range(4): ax[k + 1].imshow(fm[k], cmap="magma", vmin=-1, vmax=1); ax[k + 1].set_title(f"quantum ch {k}")
    [a.axis("off") for a in ax]; plt.tight_layout(); plt.show()

display(widgets.VBox([widgets.HBox([w2_digit, w2_idx]), widgets.HBox([w2_depth, w2_ent, w2_seed])]),
        widgets.interactive_output(show_feature_maps, dict(digit=w2_digit, idx=w2_idx, depth=w2_depth, ent=w2_ent, seed=w2_seed)))

## 3. Model playground: where does the quantum layer go?
Pick a **placement**. The notebook trains two small models on the same data:
* the **quantum** model, with a quantum layer at that spot, and
* its **classical twin**, identical except that a small classical layer sits where the quantum layer was.

That twin is the fair comparison the poster asks for. Is the quantum part doing the work, or the network around it?

| Placement | What is quantum |
|---|---|
| **input** | A *fixed* quantum filter reads image patches (quanvolution). Everything after it is classical. |
| **hidden** | A *trainable* circuit sits between classical layers. |
| **output** | A *trainable* circuit is the quantum head before the final classical layer. |

*Task: tell two handwritten digits apart (8x8 images). Try to break it: more qubits, no entanglement, fewer epochs.*

In [ ]:
class QLayer(nn.Module):
    """Trainable quantum layer: AngleEmbedding(RY) -> L x [RY, RZ per qubit + CNOT ring] -> <Z> on every qubit."""
    def __init__(self, n_q, n_layers, entangle):
        super().__init__()
        dev = qml.device("default.qubit", wires=n_q)
        @qml.qnode(dev, interface="torch", diff_method="backprop")
        def circuit(inputs, weights):
            qml.AngleEmbedding(inputs, wires=range(n_q), rotation="Y")             # ENCODE
            for l in range(n_layers):                                               # PROCESS
                for i in range(n_q):
                    qml.RY(weights[l, i, 0], wires=i); qml.RZ(weights[l, i, 1], wires=i)
                if entangle and n_q > 1:
                    for i in range(n_q): qml.CNOT(wires=[i, (i + 1) % n_q])
            return [qml.expval(qml.PauliZ(i)) for i in range(n_q)]                  # MEASURE
        self.circuit, self.n_q, self.n_layers = circuit, n_q, n_layers
        self.q = qml.qnn.TorchLayer(circuit, {"weights": (n_layers, n_q, 2)},
                                    init_method={"weights": lambda w: nn.init.normal_(w, std=0.3)})
    def forward(self, x):
        return self.q(torch.tanh(x) * (np.pi / 2)).to(x.dtype)

def build_model(placement, kind, n_q, n_layers, entangle):
    mid = QLayer(n_q, n_layers, entangle) if kind == "quantum" else nn.Sequential(nn.Tanh(), nn.Linear(n_q, n_q), nn.Tanh())
    conv = nn.Sequential(nn.Conv2d(1, 4, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Flatten())      # 8x8 -> 64 features
    if placement == "hidden":
        return nn.Sequential(conv, nn.Linear(64, n_q), mid, nn.Linear(n_q, 16), nn.ReLU(), nn.Linear(16, 2))
    if placement == "output":
        return nn.Sequential(conv, nn.Linear(64, 16), nn.ReLU(), nn.Linear(16, n_q), mid, nn.Linear(n_q, 2))
    # input: the (fixed) quantum or classical filter is applied to the data beforehand; this is the classical rest
    return nn.Sequential(nn.Conv2d(4, 4, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Flatten(), nn.Linear(16, 2))

def count_params(model):
    quantum = sum(p.numel() for m in model.modules() if isinstance(m, QLayer) for p in m.parameters())
    total = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total - quantum, quantum

In [ ]:
STATE = {}

def make_split(a, b, seed):
    mask = (LABELS == a) | (LABELS == b)
    X, y = IMGS[mask], (LABELS[mask] == b).astype(int)
    return train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)

def prepare(placement, kind, Xtr, Xte, n_layers, entangle, seed):
    if placement == "input":
        f = (lambda X: quanv_images(X, filter_weights(seed, n_layers), entangle)) if kind == "quantum" else \
            (lambda X: classical_filter_images(X, seed))
        return torch.tensor(f(Xtr), dtype=torch.float32), torch.tensor(f(Xte), dtype=torch.float32)
    return torch.tensor(Xtr, dtype=torch.float32)[:, None], torch.tensor(Xte, dtype=torch.float32)[:, None]

def train(model, Ttr, ytr, Tte, yte, epochs, lr, seed, tag, log):
    torch.manual_seed(seed); opt = torch.optim.Adam(model.parameters(), lr=lr); lossf = nn.CrossEntropyLoss()
    ytr_t, yte_t = torch.tensor(ytr), torch.tensor(yte); hist = {"loss": [], "acc": []}
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(Ttr)); tot = 0.0
        for i in range(0, len(Ttr), 32):
            idx = perm[i:i + 32]; opt.zero_grad(); loss = lossf(model(Ttr[idx]), ytr_t[idx]); loss.backward(); opt.step()
            tot += loss.item() * len(idx)
        model.eval()
        with torch.no_grad(): acc = (model(Tte).argmax(1) == yte_t).float().mean().item()
        hist["loss"].append(tot / len(Ttr)); hist["acc"].append(acc)
        log(f"  {tag:9s} epoch {ep + 1:2d}/{epochs}  loss {hist['loss'][-1]:.3f}  test accuracy {acc:.2f}")
    return hist

def run_experiment(placement, n_q, n_layers, entangle, epochs, lr, pair, seed, log):
    a, b = pair; Xtr, Xte, ytr, yte = make_split(a, b, seed); Xtr, ytr = Xtr[:250], ytr[:250]
    res, models, tens = {}, {}, {}
    for kind in ("quantum", "classical"):
        torch.manual_seed(seed)
        m = build_model(placement, kind, n_q, n_layers, entangle)
        Ttr, Tte = prepare(placement, kind, Xtr, Xte, n_layers, entangle, seed)
        res[kind] = train(m, Ttr, ytr, Tte, yte, epochs, lr, seed, kind, log)
        res[kind]["params"] = count_params(m); models[kind], tens[kind] = m, Tte
    STATE.update(models=models, tens=tens, Xte=Xte, yte=yte, pair=pair, placement=placement, n_q=n_q, n_layers=n_layers)
    return res

In [ ]:
w3_place = widgets.ToggleButtons(options=["input", "hidden", "output"], value="output", description="placement")
w3_nq = widgets.IntSlider(value=3, min=2, max=6, description="qubits", continuous_update=False)
w3_layers = widgets.IntSlider(value=2, min=1, max=4, description="circuit layers", continuous_update=False)
w3_ent = widgets.Checkbox(value=True, description="entanglement (CNOT)")
w3_epochs = widgets.IntSlider(value=6, min=1, max=15, description="epochs", continuous_update=False)
w3_lr = widgets.Dropdown(options=[0.003, 0.01, 0.03, 0.1], value=0.03, description="learn rate")
w3_pair = widgets.Dropdown(options=[("3 vs 5 (similar)", (3, 5)), ("4 vs 9 (similar)", (4, 9)), ("0 vs 1 (easy)", (0, 1)), ("1 vs 7 (medium)", (1, 7))],
                           value=(3, 5), description="digits")
w3_seed = widgets.IntSlider(value=0, min=0, max=20, description="seed", continuous_update=False)
w3_go = widgets.Button(description="Train both models", button_style="success", icon="play")
w3_out = widgets.Output()

def on_train(_):
    with w3_out:
        clear_output(wait=True)
        if w3_place.value == "input":
            print("Input placement: the quantum filter uses 4 qubits (one per pixel of a 2x2 patch) and is fixed, not trained.\n"
                  "'qubits' is ignored; 'circuit layers' and 'entanglement' still apply.\n")
        res = run_experiment(w3_place.value, w3_nq.value, w3_layers.value, w3_ent.value, w3_epochs.value, w3_lr.value,
                             w3_pair.value, w3_seed.value, log=lambda s: print(s))
        clear_output(wait=True)
        fig, ax = plt.subplots(1, 2, figsize=(10, 3.2)); ep = range(1, len(res["quantum"]["acc"]) + 1)
        for kind, col in (("quantum", "#c2185b"), ("classical", "#355c9a")):
            ax[0].plot(ep, res[kind]["acc"], "-o", color=col, label=kind if kind == "quantum" else "classical twin")
            ax[1].plot(ep, res[kind]["loss"], "-o", color=col)
        ax[0].set_title("test accuracy"); ax[0].set_ylim(0, 1.02); ax[0].legend(); ax[1].set_title("training loss")
        for a_ in ax: a_.set_xlabel("epoch")
        plt.tight_layout(); plt.show()
        for kind in ("quantum", "classical"):
            c, q = res[kind]["params"]
            print(f"{kind:9s} final test accuracy {res[kind]['acc'][-1]:.2f}   trainable params: {c} classical + {q} quantum")
        if w3_place.value != "input":
            try:
                lay = STATE["models"]["quantum"]
                lay = [m for m in lay.modules() if isinstance(m, QLayer)][0]
                print("\nThe quantum layer you just trained:\n" + qml.draw(lay.circuit)(torch.zeros(lay.n_q), torch.zeros(lay.n_layers, lay.n_q, 2)))
            except Exception as e:
                print("(circuit drawing skipped:", e, ")")
        print("\nSimulated qubits, tiny data. Read differences of a few points as noise, not as an advantage.")

w3_go.on_click(on_train)
display(widgets.VBox([w3_place, widgets.HBox([w3_nq, w3_layers, w3_ent]), widgets.HBox([w3_epochs, w3_lr, w3_seed]), w3_pair, w3_go, w3_out]))

## 4. Look at individual predictions
After training in section 3, slide through the test digits and compare what the quantum model and its classical twin each predict.

In [ ]:
w4_idx = widgets.IntSlider(value=0, min=0, max=100, description="test digit #", continuous_update=False)

def show_prediction(idx):
    if not STATE: print("Train the models in section 3 first."); return
    n = len(STATE["yte"]); idx = idx % n; a, b = STATE["pair"]
    fig, ax = plt.subplots(1, 3, figsize=(9, 2.6), gridspec_kw={"width_ratios": [1, 1.4, 1.4]})
    ax[0].imshow(STATE["Xte"][idx], cmap="gray"); ax[0].axis("off")
    ax[0].set_title(f"true: {[a, b][STATE['yte'][idx]]}")
    for k, kind in enumerate(("quantum", "classical"), start=1):
        m = STATE["models"][kind]; m.eval()
        with torch.no_grad(): p = torch.softmax(m(STATE["tens"][kind][idx:idx + 1]), 1)[0].numpy()
        ax[k].bar([str(a), str(b)], p, color="#c2185b" if kind == "quantum" else "#355c9a"); ax[k].set_ylim(0, 1)
        ax[k].set_title(f"{kind}: says {[a, b][int(p.argmax())]}")
    plt.tight_layout(); plt.show()

display(w4_idx, widgets.interactive_output(show_prediction, dict(idx=w4_idx)))

## Things to try (and what they teach)
1. **Fair baseline.** Train a few times with different *seeds*. Does the quantum model ever beat its classical twin by more than the seed-to-seed wobble?
2. **Entanglement off.** Untick the CNOT box in section 3. Does accuracy change? (Published benchmarks found that removing entanglement often made no difference.)
3. **Placement.** Same settings at *input*, *hidden*, *output*. Which one trains most smoothly?
4. **More qubits or layers.** Does a bigger circuit help, or just slow things down? (Deeper circuits can also be harder to train.)
5. **Hard digits.** Switch from 0 vs 1 to 4 vs 9. Where do both models struggle?

### Honest notes
* All qubits are **simulated** on a normal CPU, and the data is tiny (8x8 digits, 250 training images).
* The poster's feature-map figure comes from section 2 with its default settings.
* This is a teaching tool. See the poster's references for what the research actually shows.

*Questions or collaboration? Contact details are on the poster.*